In [1]:
import ee
import geemap

PROJECT_ID = "urbaninfo-510709"

print("Project ID:", PROJECT_ID)

Project ID: urbaninfo-510709


In [2]:
ee.Authenticate()

True

In [3]:
ee.Initialize(project=PROJECT_ID)

print("Google Earth Engine initialized successfully.")

*** Earth Engine *** Share your feedback by taking our Annual Developer Satisfaction Survey: https://google.qualtrics.com/jfe/form/SV_9oS0DRcPvElRMNw?source=python


Google Earth Engine initialized successfully.


In [4]:
message = ee.String("Connected to Google Earth Engine")

print(message.getInfo())

Connected to Google Earth Engine


In [7]:
!pip install -q earthengine-api geemap geopy geopandas matplotlib pandas

In [8]:

# ============================================================
# CLASS EXERCISE 20
# Urban Thermal Stress and Population Exposure Using UTCI
# Study area: Bhubaneswar, Odisha, India
# Platform: Google Colab + Google Earth Engine
# ============================================================

import ee
import geemap
import geopy
import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt
import json
import os

from geopy.geocoders import Nominatim
from shapely.geometry import mapping

# ------------------------------------------------------------
# 1. INITIALIZE GOOGLE EARTH ENGINE
# ------------------------------------------------------------

try:
    ee.Initialize()
except Exception:
    ee.Authenticate()
    ee.Initialize()

print("Google Earth Engine initialized successfully.")

# ------------------------------------------------------------
# 2. ANALYSIS SETTINGS
# ------------------------------------------------------------

CITY = "Bhubaneswar"
STATE = "Odisha"
YEAR = 2022

# Warm season for an Indian city
START_MONTH = 3
END_MONTH = 5

# Strong heat stress threshold
UTCI_THRESHOLD = 32.0

# WorldPop population year
POP_YEAR = 2021

OUTPUT_DIR = "/content/Bhubaneswar_UTCI_Results"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ------------------------------------------------------------
# 3. OBTAIN BHUBANESWAR CITY BOUNDARY
# ------------------------------------------------------------
# This obtains a geocoded boundary/area from OpenStreetMap.
# Inspect the resulting boundary before interpreting results.
# For official results, replace this with the municipal
# corporation boundary shapefile supplied by your instructor.

geolocator = Nominatim(user_agent="bhubaneswar_utci_analysis")

location = geolocator.geocode(
    "Bhubaneswar, Odisha, India",
    exactly_one=True,
    geometry="geojson",
    addressdetails=True
)

if location is None:
    raise ValueError(
        "City boundary could not be obtained. "
        "Use an official Bhubaneswar municipal boundary shapefile."
    )

# Request the detailed OSM GeoJSON boundary using Nominatim
import requests

params = {
    "q": "Bhubaneswar, Odisha, India",
    "format": "jsonv2",
    "polygon_geojson": 1,
    "limit": 1
}

response = requests.get(
    "https://nominatim.openstreetmap.org/search",
    params=params,
    headers={"User-Agent": "BhubaneswarUTCIClassExercise/1.0"},
    timeout=60
)
response.raise_for_status()
results = response.json()

if not results or "geojson" not in results[0]:
    raise ValueError(
        "No polygon returned. Load your official city boundary instead."
    )

city_gdf = gpd.GeoDataFrame(
    {"city": [CITY]},
    geometry=[gpd.GeoSeries.from_geojson(
        json.dumps(results[0]["geojson"])
    ).iloc[0]],
    crs="EPSG:4326"
)

city_geometry = city_gdf.geometry.iloc[0]

# Convert boundary into an Earth Engine geometry
city_boundary = ee.Geometry(mapping(city_geometry))

print("City:", CITY)
print("Boundary area (km2):",
      round(city_gdf.to_crs("EPSG:32645").area.iloc[0] / 1e6, 2))

# Save boundary
city_gdf.to_file(
    f"{OUTPUT_DIR}/Bhubaneswar_Boundary.geojson",
    driver="GeoJSON"
)

# ------------------------------------------------------------
# 4. LOAD GloUTCI-M DATASET
# ------------------------------------------------------------

utci_collection = ee.ImageCollection(
    "projects/sat-io/open-datasets/gloutci-m"
)

# Select March, April and May for the chosen year.
# filterDate's end date is exclusive.
start_date = f"{YEAR}-{START_MONTH:02d}-01"
end_date = f"{YEAR}-06-01"

utci_season = (
    utci_collection
    .filterDate(start_date, end_date)
    .filterBounds(city_boundary)
)

print("Number of UTCI monthly images:", utci_season.size().getInfo())

# Inspect available band names before processing
first_image = ee.Image(utci_season.first())
print("UTCI bands:", first_image.bandNames().getInfo())

# Use the first band automatically
utci_band = ee.String(first_image.bandNames().get(0))

# Calculate mean of the three monthly UTCI images
# Stored values must be divided by 100 to obtain degrees C.
utci_mean = (
    utci_season
    .select(ee.List([utci_band]))
    .mean()
    .divide(100)
    .rename("UTCI_C")
    .clip(city_boundary)
)

print("Seasonal mean UTCI image prepared.")

# ------------------------------------------------------------
# 5. CREATE HIGH THERMAL STRESS MASK
# ------------------------------------------------------------

high_stress = (
    utci_mean
    .gte(UTCI_THRESHOLD)
    .rename("High_Stress")
    .selfMask()
)

print("High-stress threshold:", UTCI_THRESHOLD, "°C")

# ------------------------------------------------------------
# 6. LOAD WORLDPOP POPULATION DATA
# ------------------------------------------------------------

population_collection = ee.ImageCollection(
    "WorldPop/GP/100m/pop"
)

# Filter to India and the selected population year.
# Check that the collection returns an image before continuing.
population_images = (
    population_collection
    .filter(ee.Filter.eq("country", "IND"))
    .filter(ee.Filter.eq("year", POP_YEAR))
    .filterBounds(city_boundary)
)

print("WorldPop images found:", population_images.size().getInfo())

if population_images.size().getInfo() == 0:
    raise ValueError(
        "WorldPop image not found. Inspect collection properties "
        "and available years in the Earth Engine catalog."
    )

population = (
    population_images
    .mosaic()
    .select("population")
    .clip(city_boundary)
)

# ------------------------------------------------------------
# 7. CALCULATE TOTAL POPULATION
# ------------------------------------------------------------
# Population band values are population counts per grid cell.
# Use sum, not mean, for population totals.

population_stats = population.reduceRegion(
    reducer=ee.Reducer.sum(),
    geometry=city_boundary,
    scale=100,
    maxPixels=1e10,
    tileScale=4
)

total_population = ee.Number(
    population_stats.get("population")
)

# ------------------------------------------------------------
# 8. CALCULATE POPULATION EXPOSED TO HIGH THERMAL STRESS
# ------------------------------------------------------------

exposed_population_image = (
    population
    .updateMask(utci_mean.gte(UTCI_THRESHOLD))
    .rename("Exposed_Population")
)

exposed_stats = exposed_population_image.reduceRegion(
    reducer=ee.Reducer.sum(),
    geometry=city_boundary,
    scale=100,
    maxPixels=1e10,
    tileScale=4
)

exposed_population = ee.Number(
    exposed_stats.get("Exposed_Population")
)

exposure_percentage = (
    exposed_population.divide(total_population).multiply(100)
)

# ------------------------------------------------------------
# 9. CALCULATE CITY-WIDE MEAN UTCI
# ------------------------------------------------------------

mean_utci_stats = utci_mean.reduceRegion(
    reducer=ee.Reducer.mean(),
    geometry=city_boundary,
    scale=1000,
    maxPixels=1e10,
    tileScale=4
)

mean_utci = ee.Number(mean_utci_stats.get("UTCI_C"))

# Calculate area above the UTCI threshold
high_stress_area = (
    ee.Image.pixelArea()
    .updateMask(utci_mean.gte(UTCI_THRESHOLD))
    .reduceRegion(
        reducer=ee.Reducer.sum(),
        geometry=city_boundary,
        scale=1000,
        maxPixels=1e10,
        tileScale=4
    )
    .get("area")
)

high_stress_area_km2 = ee.Number(high_stress_area).divide(1e6)

# ------------------------------------------------------------
# 10. DISPLAY RESULTS
# ------------------------------------------------------------

results = ee.Dictionary({
    "City": CITY,
    "UTCI_Year": YEAR,
    "Season": "March-May",
    "Population_Year": POP_YEAR,
    "Mean_UTCI_C": mean_utci,
    "Total_Population": total_population,
    "Population_Exposed": exposed_population,
    "Exposure_Percentage": exposure_percentage,
    "High_Stress_Area_km2": high_stress_area_km2
}).getInfo()

results_df = pd.DataFrame(
    [{"Parameter": k, "Value": v} for k, v in results.items()]
)

print("\n========== BHUBANESWAR UTCI RESULTS ==========")
print(results_df.to_string(index=False))

results_df.to_csv(
    f"{OUTPUT_DIR}/Bhubaneswar_UTCI_Statistics.csv",
    index=False
)

# ------------------------------------------------------------
# 11. CREATE INTERACTIVE MAP
# ------------------------------------------------------------

m = geemap.Map()
m.centerObject(city_boundary, 10)

# UTCI map
m.addLayer(
    utci_mean,
    {
        "min": 20,
        "max": 40,
        "palette": [
            "313695", "4575b4", "74add1", "abd9e9",
            "ffffbf", "fdae61", "f46d43", "d73027", "a50026"
        ]
    },
    "Seasonal Mean UTCI (°C)"
)

# High thermal stress
m.addLayer(
    high_stress,
    {"palette": ["red"]},
    "High Thermal Stress: UTCI >= 32°C"
)

# Population distribution
m.addLayer(
    population,
    {
        "min": 0,
        "max": 100,
        "palette": ["ffffff", "ffffb2", "fecc5c", "fd8d3c", "bd0026"]
    },
    f"WorldPop Population {POP_YEAR}"
)

# Boundary
m.addLayer(
    ee.FeatureCollection([ee.Feature(city_boundary)]),
    {"color": "black", "fillColor": "00000000"},
    "Bhubaneswar Boundary"
)

m.addLayerControl()
display(m)

# ------------------------------------------------------------
# 12. EXPORT MAP RASTERS TO GOOGLE DRIVE
# ------------------------------------------------------------
# These exports run as Earth Engine tasks.
# Start the tasks, then check the Tasks panel in Earth Engine
# or the task status printed below.

def export_to_drive(image, description, filename, scale):
    task = ee.batch.Export.image.toDrive(
        image=image,
        description=description,
        folder="Bhubaneswar_UTCI",
        fileNamePrefix=filename,
        region=city_boundary,
        scale=scale,
        maxPixels=1e13,
        fileFormat="GeoTIFF"
    )
    task.start()
    print("Started export:", description, "| Task ID:", task.id)
    return task

task_utci = export_to_drive(
    utci_mean,
    "Bhubaneswar_Mean_UTCI",
    "Bhubaneswar_Mean_UTCI",
    1000
)

task_stress = export_to_drive(
    high_stress.unmask(0),
    "Bhubaneswar_High_Stress",
    "Bhubaneswar_High_Stress",
    1000
)

task_population = export_to_drive(
    population,
    "Bhubaneswar_Population",
    "Bhubaneswar_Population",
    100
)

task_exposure = export_to_drive(
    exposed_population_image.unmask(0),
    "Bhubaneswar_Exposed_Population",
    "Bhubaneswar_Exposed_Population",
    100
)

# ------------------------------------------------------------
# 13. CREATE A RESULTS BAR CHART
# ------------------------------------------------------------

chart_values = [
    results["Total_Population"],
    results["Population_Exposed"]
]

plt.figure(figsize=(8, 5))
plt.bar(
    ["Total population", "Population exposed to UTCI >= 32°C"],
    chart_values
)
plt.ylabel("Population (people)")
plt.title("Bhubaneswar: Population Exposure to Thermal Stress")
plt.xticks(rotation=10)
plt.tight_layout()
plt.savefig(
    f"{OUTPUT_DIR}/Bhubaneswar_Population_Exposure.png",
    dpi=300
)
plt.show()

print("\nResults saved to:", OUTPUT_DIR)
print("Check Google Drive for exported GeoTIFF files.")


Google Earth Engine initialized successfully.


AttributeError: type object 'GeoSeries' has no attribute 'from_geojson'